In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
import os
from pathlib import Path
import matplotlib.pyplot as plt

save_path = "CES_pkl_files"
data = {}
for file in os.listdir(save_path):
    if file.endswith(".pkl"):
        name = file[:-4]
        data[name] = pd.read_pickle(os.path.join(save_path,file))
        print(f"Loaded {name}")

omega = data["omega"]
bea_prices = data["bea_prices"]
bea_expenditures = data["bea_expenditures"]
bea_quantities = data["bea_quantities"]
bea_PQE_merged = data["bea_PQE_merged"]
delta_final = data["delta_final"]

Loaded omega
Loaded bea_prices
Loaded bea_expenditures
Loaded bea_quantities
Loaded bea_PQE_merged
Loaded delta_final


## CES additional data cleaning

Basic pre-processing

In [2]:
"""Borrowing Tony's omega data, but change how 0s are cleaned and processed based on CES_Production_Take_Two by Nick"""
omega_CES = omega.copy()
products = omega_CES.index.tolist()
omega_CES = omega_CES.reindex(index = products, columns = products)

print("shape:", omega_CES.shape)

price_panel = (bea_prices.pivot_table(index="date", columns="products", values="Prices", aggfunc="first").sort_index())
price_panel = price_panel.reindex(columns = products)

omega_CES

shape: (210, 210)


,accessories and parts,air transportation,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,bakery products,beef and veal,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
accessories and parts,0.194989,0.000993,0.007536,0.000913,0.004572,0.003761,0.020231,0.000706,0.002743,0.000530,...,0.004964,0.003475,0.000257,0.000400,0.001459,0.002302,0.001490,0.001043,0.001841,0.000967
air transportation,0.012067,0.000963,0.016294,0.000755,0.051315,0.000867,0.002162,0.000264,0.001103,0.000192,...,0.005569,0.002558,0.000090,0.000097,0.006277,0.059822,0.000499,0.002421,0.000827,0.001335
alcohol in purchased meals,0.012184,0.001007,0.011103,0.000526,0.009926,0.003252,0.002596,0.000679,0.008541,0.004652,...,0.009724,0.006858,0.000043,0.000158,0.003651,0.003917,0.001200,0.008348,0.000640,0.002710
all other professional medical services,0.015413,0.001311,0.026711,0.002133,0.004971,0.001196,0.002329,0.001207,0.002604,0.000348,...,0.009887,0.005568,0.000051,0.000288,0.002194,0.001843,0.001177,0.000804,0.001232,0.003615
"amusement parks, campgrounds, and related recreational services",0.023199,0.001841,0.011491,0.001059,0.011544,0.001638,0.003601,0.003638,0.002824,0.002010,...,0.010025,0.004018,0.000089,0.000238,0.003404,0.003510,0.001344,0.002749,0.001040,0.004014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
water transportation,0.024880,0.002666,0.007614,0.000804,0.023865,0.001100,0.003990,0.001696,0.001677,0.000305,...,0.006156,0.003211,0.000110,0.000158,0.005672,0.042363,0.000731,0.000410,0.000701,0.002856
window coverings,0.038151,0.001304,0.008332,0.000885,0.005078,0.001712,0.006876,0.000923,0.002338,0.000368,...,0.008411,0.002329,0.000117,0.000220,0.002600,0.006738,0.004322,0.000429,0.006133,0.001590
wine,0.029645,0.000839,0.005250,0.000726,0.004434,0.001265,0.009236,0.001100,0.007561,0.000356,...,0.006814,0.002759,0.000141,0.000240,0.001480,0.002862,0.001162,0.064483,0.001228,0.002271
women's and girls' clothing,0.028424,0.001258,0.008368,0.001055,0.008421,0.001917,0.006506,0.001522,0.002707,0.000642,...,0.007998,0.004119,0.000185,0.000305,0.002330,0.004397,0.001256,0.000679,0.008595,0.002018


In [3]:
price_panel.dtypes

products
accessories and parts                                              float64
air transportation                                                 float64
alcohol in purchased meals                                         float64
all other professional medical services                            float64
amusement parks, campgrounds, and related recreational services    float64
                                                                    ...   
water transportation                                               float64
window coverings                                                   float64
wine                                                               float64
women's and girls' clothing                                        float64
workers' compensation                                              float64
Length: 210, dtype: object

Replace non-existing prices

In [4]:
# Not adjusting actual weights, some products don't exist so we swap those for zeroes
def price_exists(df):
    return pd.notna(df) & np.isfinite(df) & df > 0
            

omega_monthly_raw = {}
for t in range(len(price_panel.index)-1):
    price_exist_tm1 = price_exists(price_panel.iloc[t]).to_numpy()
    mask = np.outer(price_exist_tm1,price_exist_tm1)
    omega_CES_monthly = omega_CES.mul(mask)


    omega_monthly_raw[price_panel.index[t+1]] = omega_CES_monthly

omega_monthly_raw[price_panel.index[1]] #starts from 1959-02-01, careful when referencing

,accessories and parts,air transportation,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,bakery products,beef and veal,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
accessories and parts,0.194989,0.000993,0.007536,0.0,0.004572,0.003761,0.020231,0.0,0.002743,0.000530,...,0.0,0.0,0.0,0.000400,0.001459,0.002302,0.001490,0.001043,0.001841,0.000967
air transportation,0.012067,0.000963,0.016294,0.0,0.051315,0.000867,0.002162,0.0,0.001103,0.000192,...,0.0,0.0,0.0,0.000097,0.006277,0.059822,0.000499,0.002421,0.000827,0.001335
alcohol in purchased meals,0.012184,0.001007,0.011103,0.0,0.009926,0.003252,0.002596,0.0,0.008541,0.004652,...,0.0,0.0,0.0,0.000158,0.003651,0.003917,0.001200,0.008348,0.000640,0.002710
all other professional medical services,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"amusement parks, campgrounds, and related recreational services",0.023199,0.001841,0.011491,0.0,0.011544,0.001638,0.003601,0.0,0.002824,0.002010,...,0.0,0.0,0.0,0.000238,0.003404,0.003510,0.001344,0.002749,0.001040,0.004014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
water transportation,0.024880,0.002666,0.007614,0.0,0.023865,0.001100,0.003990,0.0,0.001677,0.000305,...,0.0,0.0,0.0,0.000158,0.005672,0.042363,0.000731,0.000410,0.000701,0.002856
window coverings,0.038151,0.001304,0.008332,0.0,0.005078,0.001712,0.006876,0.0,0.002338,0.000368,...,0.0,0.0,0.0,0.000220,0.002600,0.006738,0.004322,0.000429,0.006133,0.001590
wine,0.029645,0.000839,0.005250,0.0,0.004434,0.001265,0.009236,0.0,0.007561,0.000356,...,0.0,0.0,0.0,0.000240,0.001480,0.002862,0.001162,0.064483,0.001228,0.002271
women's and girls' clothing,0.028424,0.001258,0.008368,0.0,0.008421,0.001917,0.006506,0.0,0.002707,0.000642,...,0.0,0.0,0.0,0.000305,0.002330,0.004397,0.001256,0.000679,0.008595,0.002018


In [5]:
max_row = -9999999.0
max_col = -9999999.0 #scuffed but works fine enough

for date in omega_monthly_raw:
    max_row = max(max_row, omega_monthly_raw[date].sum(axis=1).max())
    max_col = max(max_col, omega_monthly_raw[date].sum(axis=0).max())

print("Max row:", max_row)
print("Max col:", max_col)

# use A2 = A' for all future steps, because columns must sum to around 1 (old A1 had rows summing to 1 hence we transpose)

Max row: 0.9999534691349633
Max col: 21.716715738064174


In [6]:
# no longer touch original omega_monthly_raw

A_monthly = {date:matrix.T for date, matrix in omega_monthly_raw.items()}

## Production Function Derivation

Calculate Sales (Steps 4-5)

In [7]:
# does BEA expenditure match PCE at all? - yes, it seems it has the same number of unique products/categories fortunately

sales_monthly = {}

for date in A_monthly.keys():
    pce_expenditure_t = (bea_expenditures.loc[bea_expenditures["date"] == date].set_index("products")["Expenditures"])
    A_t = A_monthly[date]
    I = np.eye(len(A_t))
    pce_expenditure_t = pce_expenditure_t.reindex(A_t.index, fill_value=0)
    assert pce_expenditure_t.index.equals(A_t.index)

    sales_t = np.linalg.solve(I - A_t.T.to_numpy(), pce_expenditure_t.to_numpy())
    # To get around Det = 0, we solve for "[I - A_t.T.to_numpy()] @ sales_t = PC_i,t" instead

    sales_monthly[date] = pd.Series(sales_t, index = A_t.index, name = date)

In [8]:
for date in sales_monthly:
    bad = sales_monthly[date] < 0
    if bad.any():
        print(date)
        print(sales_monthly[date][bad])
        break

Calculate B (Step 6-9)

In [9]:
epsilon = 0.3 # arbitrary number

pi_ela_monthly = {}
for t in range (1, len(price_panel.index)):
    old_time = price_panel.index[t-1]
    new_time = price_panel.index[t]
    pi_t = (price_panel.loc[new_time] / price_panel.loc[old_time]) ** (1-epsilon)

    pi_ela_monthly[new_time] = pi_t


In [10]:
"unfinished do not proceed further than step 9"
B_monthly = {}

for date in A_monthly.keys():
    A_t = A_monthly[date]
    I = np.eye(len(A_t))

    # separated so I don't get too dizzy
    Leontief_inv = I - A_t.to_numpy()
    
    col_sums = A_t.to_numpy().sum(axis=0)
    diag_matrix = np.diag(1 - col_sums)

    B_t = np.linalg.solve(Leontief_inv, diag_matrix)
    B_monthly[date] = B_t

B_monthly[next(iter(B_monthly))]



array([[0.09028558, 0.02920352, 0.02816396, ..., 0.02114493, 0.03009334,
        0.02016465],
       [0.00121129, 0.07418959, 0.00159003, ..., 0.00110322, 0.00161662,
        0.00114062],
       [0.00814008, 0.01099959, 0.08457465, ..., 0.00735696, 0.01078127,
        0.00735982],
       ...,
       [0.0005925 , 0.00091287, 0.00133132, ..., 0.05459226, 0.00073654,
        0.00049421],
       [0.00099935, 0.00122771, 0.00117593, ..., 0.00086714, 0.07647614,
        0.00080142],
       [0.00229496, 0.00299074, 0.00322673, ..., 0.00222338, 0.00315165,
        0.0534939 ]], shape=(210, 210))

Check B is valid (row sums to 1)

In [11]:
for date in B_monthly:
    row_sums = B_monthly[date] @ np.ones(len(B_monthly[date]))
    col_sums = B_monthly[date].sum(axis=0)
    
    bad = abs((row_sums) -1) > 0.00001 # all row sums positive so Abs is ok
    if bad.any():
        print("B_matrix stats for", date)
        print("Min row sum:", row_sums.min(), "\nMax row sum:", row_sums.max())
        print("Offending rows:")
        print(B_monthly[date][bad])
        break

B_matrix stats for 1959-02-01 00:00:00
Min row sum: 0.06331643316378044 
Max row sum: 17.797183824649338
Offending rows:
[[0.09028558 0.02920352 0.02816396 ... 0.02114493 0.03009334 0.02016465]
 [0.00121129 0.07418959 0.00159003 ... 0.00110322 0.00161662 0.00114062]
 [0.00814008 0.01099959 0.08457465 ... 0.00735696 0.01078127 0.00735982]
 ...
 [0.0005925  0.00091287 0.00133132 ... 0.05459226 0.00073654 0.00049421]
 [0.00099935 0.00122771 0.00117593 ... 0.00086714 0.07647614 0.00080142]
 [0.00229496 0.00299074 0.00322673 ... 0.00222338 0.00315165 0.0534939 ]]


Value added inflation (Step 10)